## Training Data Preparation

Training data needs to be stored as image, outline mask, and vein mask in one folder.

Images are currently all stored in one folder, outline mask and vein mask need to be read from Label-Studio annotations JSON.  
Images and masks should also be resized to 448x448 px.

In [1]:
import json
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
import random

In [86]:
root_path = "/masc_shared/ag_nauss/projects/phytoakmeter/IEE/vein_segmentation_training_data"

annotations_filename = "annotations.json"
path = os.path.join(root_path, annotations_filename)
with open(path, "r") as f:
    annotations = json.load(f)

In [104]:
len(scan_annotations)

10

In [103]:
photo_annotations = annotations[-5:]
scan_annotations = annotations[-15:-5]

In [15]:
def resized_outline_mask(

    data: dict,
    dst_path: str|None = None,
    size: tuple[int, int] = (448, 448),
    
) -> NDArray:

    try:
        
        ann = data['annotations'][0]
        outline = next(r for r in ann['result'] if r['value'].get('polygonlabels') == ['outline'])
        
        w = outline['original_width']
        h = outline['original_height']
        pts_pct = outline['value']['points']
        
        # convert percentage coords to pixel coords
        pts = np.array([[p[0] / 100 * w, p[1] / 100 * h] for p in pts_pct], dtype=np.int32)
        
        mask = np.full((h, w), 0, dtype=np.uint8)  # black background
        cv2.fillPoly(mask, [pts], color=255)       # white foreground
        resized = cv2.resize(mask, size, interpolation=cv2.INTER_NEAREST)

        if dst_path is not None:
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            cv2.imwrite(dst_path, resized)
        
        return resized
        
    except Exception as e:
        
        print(e)

In [24]:
def resized_vein_mask(

    data: dict,
    outline_mask: NDArray,
    dst_path: str|None = None,
    size: tuple[int, int] = (448, 448),

) -> NDArray:

    try:
        ann = data['annotations'][0]
        veins = [r for r in ann['result']
                 if r['value'].get('polygonlabels') in (['midrib_polygon'], ['sec_vein_polygon'])]
        w = veins[0]['original_width']
        h = veins[0]['original_height']
        
        vein_mask = np.full((h, w), 0, dtype=np.uint8)  # black background
        
        for v in veins:
            pts_pct = v['value']['points']
            pts = np.array([[p[0] / 100 * w, p[1] / 100 * h] for p in pts_pct], dtype=np.int32)
            cv2.fillPoly(vein_mask, [pts], color=255)  # white foreground

        resized_vein_mask = cv2.resize(vein_mask, size, interpolation=cv2.INTER_NEAREST)
        assert resized_vein_mask.shape == outline_mask.shape
        # clip to the outline mask to exclude midribs that were drawn "through" other leaves obscuring the target leaf
        resized_vein_mask[outline_mask == 0] = 0
    
        if dst_path is not None:
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            cv2.imwrite(dst_path, resized_vein_mask)
        
        return resized_vein_mask

    except Exception as e:

        print(e)

In [22]:
def resized_masked_image(
    
    src_path: str,
    mask: NDArray,
    dst_path: str|None = None,
    size: tuple[int, int] = (448, 448),

) -> NDArray:

    try:
        
        src = cv2.imread(src_path)
        resized = cv2.resize(src, (448, 448), interpolation=cv2.INTER_LINEAR)
        #- keep_mask = cv2.bitwise_not(mask) # outline_mask is inverted, so inverted back here
        masked = cv2.bitwise_and(resized, resized, mask=mask)
        masked[mask == 0] = 255 # inverted here, because model expects white background

        if dst_path is not None:
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            cv2.imwrite(dst_path, masked)
        
        return masked
        
    except Exception as e:
        
        print(e)

In [105]:
for ann in scan_annotations:
    print(ann["file_upload"])

604a7a67-8_9_2A_M_1_leaf_02.jpg
a6b9104c-8_9_2A_M_2_leaf_16.jpg
dd576844-8_9_2A_M_2_leaf_17.jpg
662c02b8-104_105_15D_LM_1_leaf_5.jpg
706e8fca-104_105_15D_LM_1_leaf_08.jpg
02758b59-104_105_15D_LM_3_leaf_4.jpg
166e8ca8-104_105_15D_LM_3_leaf_09.jpg
388de1f9-135_136_20A_M_1_leaf_01.jpg
8fb4e875-135_136_20A_M_1_leaf_07.jpg
c7367209-135_136_20A_M_1_leaf_11.jpg


In [108]:
k_pretrain, k_val = 7, 2 # 25 annotations, 15 used for pretraining, 5 for validation, 5 for testing

idx_all = list(range(len(scan_annotations)))

idx_pretrain = random.sample(idx_all, k_pretrain)
remaining = [i for i in idx_all if i not in idx_pretrain]

idx_val = random.sample(remaining, k_val)
idx_test = [i for i in remaining if i not in idx_val]

partitions = {
    "partition": ["pretrain", "valid", "test"],
    "indices":[idx_pretrain, idx_val, idx_test]
   }

In [109]:
partitions

{'partition': ['pretrain', 'valid', 'test'],
 'indices': [[0, 5, 2, 9, 1, 4, 8], [7, 6], [3]]}

In [113]:
dst_size = (448, 448)
dataset = "scan"

for partition, idx in zip(partitions["partition"], partitions["indices"]):

    partition_path = os.path.join(root_path, dataset+"_labels", partition)
    partition_annotations = [scan_annotations[i] for i in idx]

    for annotation in partition_annotations:
        
        file_upload = annotation["file_upload"].split("-")[1]
        leaf_id = os.path.splitext(file_upload)[0]
        leaf_dir = os.path.join(partition_path, leaf_id)
    
        outline_dst_path = os.path.join(leaf_dir, "outline.png")
        outline_mask = resized_outline_mask(annotation, dst_path=outline_dst_path, size=dst_size)
    
        vein_dst_path = os.path.join(leaf_dir, "vein.png")
        resized_vein_mask(annotation, outline_mask, dst_path=vein_dst_path, size=dst_size)
        
        img_src_path = os.path.join("/masc_shared/ag_nauss/projects/phytoakmeter/IEE/scan_cutouts", file_upload.replace("LM", "L+M"))
        img_dst_path = os.path.join(leaf_dir, "leaf.png")
        resized_masked_image(img_src_path, outline_mask, dst_path=img_dst_path, size=dst_size)